In [3]:
%cd ..
import gradio as gr
import numpy as np
from PIL import Image
import asyncio
from utils.requests import Request, Response
from utils.file import File
import uuid
import websockets
from websockets.asyncio.client import connect
import json
from pathlib import Path

d:\StyleTransferAI\StyleTransferAI_AdaIN


### TODO
- Transmit all interface components as generation function input to access their real value
- Handle request cancellation
- Use Requests and Responses objects to communicate

In [ ]:
MAX_STYLES = 5 
IMG_HEIGHT = 300
API_URL = "ws://localhost:8000/generate"

class StyleBlock :
    def __init__(self, visible = False):
        with gr.Group(visible=visible) as self.block:
            self.img = gr.Image(interactive=True, height=300, type='filepath')
            self.weight = gr.Slider(label="Weight", minimum=0, maximum=1, value=1, interactive=True)
            self.scale = gr.Slider(label="Spatial scaling", minimum=0.1, maximum=3, value=1, interactive=True)
            self.rmv_btn = gr.Button(value="Remove style", visible=False)
            self.true_res_img = gr.Image(render=False)
        # self.img.input(self.resize, inputs=self.img, outputs=self.img) # Image is always resized to a fixed height to make sure all style blocks are correctly aligned

    def resize(self, img) :
        print(img)
        self.true_res_img.value = img # Keep the original image as it will be used for computation
        if img is None :
            return gr.update(value=None)
        # img = Image.fromarray(img)
        img = Image.fr
        new_width = int((IMG_HEIGHT / img.height) * img.width)
        new_img = img.resize((new_width, IMG_HEIGHT))
        return gr.update(value=new_img)

    def list(self) :
        return [self.img, self.weight, self.scale, self.true_res_img]
    
    def reset(self) :
        return (gr.update(value=None), gr.update(value=1), gr.update(value=1), gr.update(value=None))
    
    def copy(self, new_img, new_weight, new_scale, new_true_res_img) :
        return (gr.update(value = new_img), gr.update(value = new_weight), gr.update(value = new_scale), gr.update(value = new_true_res_img))


class ParamsBlock :
    def __init__(self):
        with gr.Blocks() as self.block :
            self.alpha = gr.Slider(0, 1, value = 1, label="Importance of stylization", interactive=True)
            with gr.Row(equal_height=True) :
                with gr.Column(scale=2) :
                    self.pres_col = gr.Checkbox(label="Preserve colors", value=False)
                    # sep = gr.Markdown("---")
                    with gr.Group() :
                        self.res_size = gr.Slider(label="Resize size", minimum=256, maximum=2000, value=1000, interactive=True)
                        self.keep_ratio = gr.Checkbox(label="Keep aspect ratio", value=False)
                with gr.Column(scale=1) :
                    self.patches = gr.Checkbox(label="Work with patches", value=False)
                    self.patch_size = gr.Slider(label="Patch size", minimum=256, maximum=1000, interactive=False)
                    self.patch_context_size = gr.Slider(label="Patch context size", minimum=256, maximum=1500, interactive=False)
                    self.patch_overlap = gr.Slider(label="Patch overlap", minimum=0, maximum=0.9, value=0.5, interactive=False)
                self.patches.change(self.update_patches_params, inputs=self.patches, outputs=[self.patch_size, self.patch_context_size, self.patch_overlap])
    
    def update_patches_params(self, enable : bool) :
        return gr.update(interactive=enable), gr.update(interactive=enable), gr.update(interactive=enable)
    
    def get_params_dict(self) :
        return {
            "alpha" : self.alpha.value,
            "preserve_colors" : self.pres_col.value,
            "resize_size" : self.res_size.value,
            "keep_aspect_ratio" : self.keep_ratio.value,
            "work_with_patches" : self.patches.value,
            "patch_size" : self.patch_size.value,
            "patch_context_size" : self.patch_context_size.value,
            "patch_overlap" : self.patch_overlap.value
        }
                

class GenConfigBlock :
    def __init__(self, scale) :
        with gr.Blocks() as self.block :
            self.index_state = gr.State(1)
            self.style_blocks = []
            self.add_btn = gr.Button(value="➕ Add style", render=False)    
            with gr.Column(scale=scale) :
                self.content_img = gr.Image(type='filepath')
                with gr.Group() :
                    with gr.Row(equal_height=True) as row :
                        for i in range(MAX_STYLES):
                            self.style_blocks.append(StyleBlock(visible = (i == 0)))
                        self.link_rmv_btns()
                        self.add_btn.render() 
                        self.add_btn.click(self.show_next_block, inputs=self.index_state, outputs=[b.block for b in self.style_blocks] + [b.rmv_btn for b in self.style_blocks] + [self.index_state, self.add_btn])
                        
                self.params = ParamsBlock()

    def link_rmv_btns(self) :
        for i, block in enumerate(self.style_blocks) :
            block.rmv_btn.click(self.hide_block, inputs=[gr.State(i), self.index_state]  + [b.img for b in self.style_blocks] + [b.weight for b in self.style_blocks] + [b.scale for b in self.style_blocks] + [b.true_res_img for b in self.style_blocks], outputs = [b.block for b in self.style_blocks] + [self.add_btn] + [b.rmv_btn for b in self.style_blocks] + [item for b in self.style_blocks for item in b.list()] + [self.index_state])

    def hide_block(self, index, index_state, *blocks_values) :
        new_index = index_state - 1
        print(new_index)
        img_values = blocks_values[:MAX_STYLES]
        weight_values = blocks_values[MAX_STYLES:2*MAX_STYLES]
        scale_values = blocks_values[2*MAX_STYLES:3*MAX_STYLES]
        true_res_imgs = blocks_values[3*MAX_STYLES:4*MAX_STYLES]
        visible_updates = [gr.update(visible=True) if i < new_index else gr.update(visible=False) for i in range(MAX_STYLES)]
        add_btn_update = gr.update(visible=(new_index < MAX_STYLES))
        rmv_btns_updates = [gr.update(visible=False) for _ in range(MAX_STYLES)] if new_index == 1 else [gr.update(visible=True) for _ in range(MAX_STYLES)]
        copy_updates = [[gr.update() for _ in range(4)] for _ in range(MAX_STYLES)]
        for i in range(index, MAX_STYLES - 1) :
            copy_updates[i] = self.style_blocks[i].copy(img_values[i+1], weight_values[i+1], scale_values[i+1], true_res_imgs[i+1])
        copy_updates[-1] = self.style_blocks[-1].reset()
        return (*visible_updates, add_btn_update, *rmv_btns_updates, *[item for cp_update in copy_updates for item in cp_update], new_index)

    def show_next_block(self, index) :
        new_index = index + 1
        print(new_index)
        updates = [gr.update(visible = (i <= index)) for i in range(MAX_STYLES)]
        button_update = gr.update(visible = (new_index < MAX_STYLES))
        if new_index == 1 :
            rmv_btns_updates = [gr.update(visible=False) for _ in range(MAX_STYLES)]
        else : 
            rmv_btns_updates = [gr.update(visible=True) for _ in range(MAX_STYLES)]
        return (*updates, *rmv_btns_updates, new_index, button_update)
    
    def get_params_dict(self) :
        global_params = self.params.get_params_dict()
        style_weights = [b.weight.value for b in self.style_blocks if b.block.visible and b.img.value is not None]
        style_scales = [b.scale.value for b in self.style_blocks if b.block.visible and b.img.value is not None]

        return {
            "style_weights" : style_weights,
            "style_scales" : style_scales,
            **global_params
        }


class MainInterface :
    def __init__(self):
        self.client_id = uuid.uuid4().hex
        with gr.Blocks() as self.interface :
            with gr.Row(equal_height=False) :
                self.config = GenConfigBlock(scale=6)
                with gr.Column(scale=4) : 
                    self.generated = gr.Image(type='filepath', interactive=False)
                    self.gen_btn = gr.Button(value="Generate image")
                    self.gen_status = gr.Text(interactive=False, label="Status")
                    self.gen_msg = gr.Text(interactive=False, label="Message")
            self.gen_btn.click(self.generate_img, inputs=[self.config.content_img] + [b.img for b in self.config.style_blocks if b.block.visible], outputs=[self.generated, self.gen_status, self.gen_msg])
    
    def launch(self) :
        self.interface.launch()
    
    async def generate_img(self, *imgs) :
        content_img = Path(imgs[0])
        print(content_img)
        style_imgs = [Path(style_img) for style_img in imgs[1:] if style_img is not None]
        print(style_imgs)
        params = self.config.get_params_dict()
        print(params)
        async with connect(API_URL, max_size=5*1024*1024) as websocket :
            # response = await request_generation(websocket, self.client_id, content_img, style_imgs, params)
            request = {
                "client_id" : self.client_id,
                "content" : File.from_path(content_img).model_dump(),
                "style" : [File.from_path(style_img).model_dump() for style_img in style_imgs],
                "params" : params
            }
            await websocket.send(json.dumps(request))
            while True:
                try:
                    response = await websocket.recv()
                    msg = json.loads(response)
                    status_update = gr.update(value=msg.get("status"))
                    msg_update = gr.update(value=msg.get("message"))
                    if msg.get("status") in ("success"):
                        gen_file = File.from_dict(msg["generated_image"])
                        gen_img_path = gen_file.save_to(Path('D:/StyleTransferAI/StyleTransferAI_AdaIN/tmp/resp'))
                    gen_img_update = gr.update(value=gen_img_path)
                    return (gen_img_update, status_update, msg_update)
                except websockets.ConnectionClosed:
                    return (gr.update(value=None), gr.update(value="Connection closed"), gr.update(value=None))

async def request_generation(connection, client_id, content_img, style_imgs, params) :
    request = Request(client_id=client_id, content_img=content_img, style_imgs=style_imgs, params=params)
    await connection.send(request.model_dump())
    response = await Response(**connection.recv())
    return Response

interface = MainInterface()
interface.launch()
interface.config.content_img.value

* Running on local URL:  http://127.0.0.1:7873

To create a public link, set `share=True` in `launch()`.


C:\Users\yoanb\AppData\Local\Temp\gradio\672555e0689c6a69420dd00f796d00a910455acc86474af0bf029665d2daca32\Capture d'écran 2024-04-17 203938.png
[WindowsPath("C:/Users/yoanb/AppData/Local/Temp/gradio/3c6d1e0ea30dd4236b4ce80c5aa869c5d5a11bcd14146982c6eadf4f26994b70/Capture d'écran 2024-07-01 112734.png")]
{'style_weights': [1], 'style_scales': [1], 'alpha': 1, 'preserve_colors': False, 'resize_size': 1000, 'keep_aspect_ratio': False, 'work_with_patches': False, 'patch_size': 256, 'patch_context_size': 256, 'patch_overlap': 0.5}
C:\Users\yoanb\AppData\Local\Temp\gradio\0cc975ddd71ccd4a0e6315a03b6403c7f7537f6ea6a408193219c46a3a39e0a4\city.jpg
[WindowsPath('C:/Users/yoanb/AppData/Local/Temp/gradio/2194108c0b21f2cbf2221d8a5bda3ac49c819dc6cb3b70722038ca4c8698d462/croquis.jpg')]
{'style_weights': [1], 'style_scales': [1], 'alpha': 1, 'preserve_colors': False, 'resize_size': 1000, 'keep_aspect_ratio': False, 'work_with_patches': False, 'patch_size': 256, 'patch_context_size': 256, 'patch_overla

In [17]:
i = gr.Image()
i.path

AttributeError: 'Image' object has no attribute 'path'

In [ ]:
MAX_STYLES = 5 
IMG_HEIGHT = 300
class StyleBlock :
    def __init__(self, visible = False):
        with gr.Group(visible=visible) as block:
            self.img = gr.Image(interactive=True, height=300)
            self.weight = gr.Slider(label="Weight", minimum=0, maximum=1, value=1, interactive=True)
            self.scale = gr.Slider(label="Spatial scaling", minimum=0.1, maximum=3, value=1, interactive=True)
            self.rmv_btn = gr.Button(value="Remove style", visible=False)
        self.block = block
        self.img.input(self.resize, inputs=self.img, outputs=self.img) # Image is always resized to a fixed height to make sure all style blocks are correctly aligned

    def resize(self, img) :
        self.true_res_img = img # Keep the original image as it will be used for computation
        if img is None :
            return gr.update(value=None)
        img = Image.fromarray(img)
        print(img.height, img.width)
        new_width = int((IMG_HEIGHT / img.height) * img.width)
        new_img = img.resize((new_width, IMG_HEIGHT))
        return gr.update(value=new_img)

    def list(self) :
        return [self.img, self.weight, self.scale]
    
    def reset(self) :
        return (gr.update(value=None), gr.update(value=1), gr.update(value=1))
    
    def copy(self, new_img, new_weight, new_scale) :
        return (gr.update(value = new_img), gr.update(value = new_weight), gr.update(value = new_scale))

with gr.Blocks() as demo :
    style_blocks = []
    index_state = gr.State(1)
    add_btn = gr.Button(value="➕ Add style", render=False)

    def link_rmv_btns() :
        for i, block in enumerate(style_blocks) :
           block.rmv_btn.click(hide_block, inputs=[gr.State(i), index_state]  + [b.img for b in style_blocks] + [b.weight for b in style_blocks] + [b.scale for b in style_blocks], outputs = [b.block for b in style_blocks] + [add_btn] + [b.rmv_btn for b in style_blocks] + [item for b in style_blocks for item in b.list()] + [index_state])

    def hide_block(index, index_state, *blocks_values) :
        new_index = index_state - 1
        img_values = blocks_values[:MAX_STYLES]
        weight_values = blocks_values[MAX_STYLES:2*MAX_STYLES]
        scale_values = blocks_values[2*MAX_STYLES:3*MAX_STYLES]
        visible_updates = [gr.update(visible=True) if i < new_index else gr.update(visible=False) for i in range(MAX_STYLES)]
        add_btn_update = gr.update(visible=(new_index < MAX_STYLES))
        rmv_btns_updates = [gr.update(visible=False) for _ in range(MAX_STYLES)] if new_index == 1 else [gr.update(visible=True) for _ in range(MAX_STYLES)]
        copy_updates = [[gr.update() for _ in range(3)] for _ in range(MAX_STYLES)]
        for i in range(index, MAX_STYLES - 1) :
            copy_updates[i] = style_blocks[i].copy(img_values[i+1], weight_values[i+1], scale_values[i+1])
        copy_updates[-1] = style_blocks[-1].reset()
        return (*visible_updates, add_btn_update, *rmv_btns_updates, *[item for cp_update in copy_updates for item in cp_update], new_index)

    with gr.Row(equal_height=True) as row :
        for i in range(MAX_STYLES):
            style_blocks.append(StyleBlock(visible = (i == 0)))
        link_rmv_btns()
        add_btn.render()

    @add_btn.click(inputs=index_state, outputs=[b.block for b in style_blocks] + [b.rmv_btn for b in style_blocks] + [index_state, add_btn])
    def show_next_block(index) :
        new_index = index + 1
        updates = [gr.update(visible = (i <= index)) for i in range(MAX_STYLES)]
        button_update = gr.update(visible = (new_index < MAX_STYLES))
        if new_index == 1 :
            rmv_btns_updates = [gr.update(visible=False) for _ in range(MAX_STYLES)]
        else : 
            rmv_btns_updates = [gr.update(visible=True) for _ in range(MAX_STYLES)]
        return (*updates, *rmv_btns_updates, new_index, button_update)

demo.launch()

* Running on local URL:  http://127.0.0.1:7880

To create a public link, set `share=True` in `launch()`.


4000 3000
348 483
770 1723
413 186
757 2071
